# 05 K-Center Embedding Baseline

For each supported dataset, select K_PER_CLASS examples per class with k-center over BERT CLS embeddings and train the classifier.

In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from text_distillation.data import get_dataset_info, get_train_eval_splits, load_text_classification_dataset, make_tiny_subset
from text_distillation.data.datasets import get_label_names
from text_distillation.distillation import select_kcenter_embeddings
from text_distillation.model import train_text_classifier
from text_distillation.saving import create_run_dir, save_distilled_dataset, save_experiment_config, save_metrics
from text_distillation.utils import get_git_commit_hash, set_seed


In [ ]:
EXPERIMENT_PREFIX = "kcenter_cls"
K_PER_CLASS = 100
EMBEDDING_MODEL_NAME = "bert-base-uncased"
EMBEDDING_BATCH_SIZE = 128
DATASET_NAMES = ["ag_news", "sst2", "qqp", "mnli-m"]
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# T4 16GB starting point. If CUDA OOM happens, lower TRAIN_BATCH_SIZE to 32.
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
MIXED_PRECISION = "auto"
DATALOADER_NUM_WORKERS = 2
SEED = 42

MAX_EVAL_SAMPLES = None

set_seed(SEED)

# Keep None for real runs. For smoke checks, make sure this is >= K_PER_CLASS.
MAX_TRAIN_POOL_DPC = None


In [ ]:
def safe_name(value):
    return value.replace("-", "_")


def maybe_make_tiny_eval(eval_dataset):
    if MAX_EVAL_SAMPLES is None:
        return eval_dataset
    return make_tiny_subset(eval_dataset, total_size=MAX_EVAL_SAMPLES, seed=SEED)


def get_labels(train_dataset, dataset_info):
    label_names = get_label_names(train_dataset, dataset_info.label_column)
    num_labels = len(label_names) if label_names is not None else len(set(train_dataset[dataset_info.label_column]))
    return label_names, num_labels

def maybe_make_tiny_train_pool(train_dataset, dataset_info):
    if MAX_TRAIN_POOL_DPC is None:
        return train_dataset
    return make_tiny_subset(train_dataset, n_per_class=MAX_TRAIN_POOL_DPC, label_column=dataset_info.label_column, seed=SEED)


In [ ]:
def train_and_save(
    *,
    experiment_name,
    dataset_name,
    dataset_info,
    train_dataset,
    eval_dataset,
    label_names,
    num_labels,
    method_name,
    full_train_size,
    train_pool_size=None,
    dpc=None,
    k_total=None,
    selection_time_sec=0.0,
    save_dataset=True,
    extra_config=None,
):
    run_dir = create_run_dir(PROJECT_ROOT / "artifacts" / "runs", experiment_name)
    config = {
        "experiment_name": experiment_name,
        "dataset_name": dataset_name,
        "method_name": method_name,
        "model_name": MODEL_NAME,
        "dpc": dpc,
        "k_total": k_total if k_total is not None else len(train_dataset),
        "text_columns": dataset_info.text_columns,
        "label_column": dataset_info.label_column,
        "metric_name": dataset_info.metric_name,
        "train_split": dataset_info.train_split,
        "eval_split": dataset_info.eval_split,
        "max_length": MAX_LENGTH,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "mixed_precision": MIXED_PRECISION,
        "dataloader_num_workers": DATALOADER_NUM_WORKERS,
        "seed": SEED,
        "full_train_size": full_train_size,
        "train_pool_size": train_pool_size,
        "train_size": len(train_dataset),
        "eval_size": len(eval_dataset),
        "compression_ratio": full_train_size / len(train_dataset),
        "selection_time_sec": selection_time_sec,
        "git_commit": get_git_commit_hash(PROJECT_ROOT),
    }
    if extra_config:
        config.update(extra_config)
    save_experiment_config(config, run_dir)
    if save_dataset:
        save_distilled_dataset(train_dataset, run_dir)

    train_start = time.perf_counter()
    _, metrics = train_text_classifier(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        model_name=MODEL_NAME,
        output_dir=run_dir,
        text_columns=dataset_info.text_columns,
        label_column=dataset_info.label_column,
        num_labels=num_labels,
        label_names=label_names,
        max_length=MAX_LENGTH,
        metric_name=dataset_info.metric_name,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=MIXED_PRECISION,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        seed=SEED,
    )
    metrics["training_time_sec"] = time.perf_counter() - train_start
    metrics["selection_time_sec"] = selection_time_sec
    save_metrics(metrics, run_dir)
    return {**config, **metrics}


In [ ]:
results = []
for dataset_name in DATASET_NAMES:
    print(f"\n=== K-center CLS: {dataset_name} ===")
    dataset_info = get_dataset_info(dataset_name)
    dataset = load_text_classification_dataset(dataset_name)
    full_train_dataset, eval_dataset = get_train_eval_splits(dataset, dataset_name)
    train_pool = maybe_make_tiny_train_pool(full_train_dataset, dataset_info)
    eval_dataset = maybe_make_tiny_eval(eval_dataset)
    label_names, num_labels = get_labels(full_train_dataset, dataset_info)
    selection_start = time.perf_counter()
    distilled_dataset = select_kcenter_embeddings(train_pool, text_columns=dataset_info.text_columns, label_column=dataset_info.label_column, k_per_class=K_PER_CLASS, model_name=EMBEDDING_MODEL_NAME, batch_size=EMBEDDING_BATCH_SIZE, max_length=MAX_LENGTH, seed=SEED)
    selection_time_sec = time.perf_counter() - selection_start
    experiment_name = f"{EXPERIMENT_PREFIX}_{safe_name(dataset_name)}_bert_base_k{K_PER_CLASS}"
    row = train_and_save(
        experiment_name=experiment_name, dataset_name=dataset_name, dataset_info=dataset_info,
        train_dataset=distilled_dataset, eval_dataset=eval_dataset, label_names=label_names, num_labels=num_labels,
        method_name="kcenter_cls", full_train_size=len(full_train_dataset), train_pool_size=len(train_pool),
        dpc=K_PER_CLASS, k_total=len(distilled_dataset), selection_time_sec=selection_time_sec,
        extra_config={"embedding_model_name": EMBEDDING_MODEL_NAME, "embedding_batch_size": EMBEDDING_BATCH_SIZE},
    )
    results.append(row)
    print(dataset_name, {k: row[k] for k in row if k in ["score", "accuracy", "f1", "f1_macro"]})
results
